# Preflight + Read Kit (v0.1.1)

**What it does.** Before you spend GPU time reading a surface, the kit checks whether the surface sits on the papyrus properly, and shows where it doesn't. If the surface is worth reading, it reads it with Reader V3 in a fixed, tested direction and writes a one-page report.

For every surface you get:
- a **verdict**: PASS (worth reading), BORDERLINE (worth reading, check by eye) or FAIL (skip: the surface isn't on the papyrus well enough)
- a **map** of where on the page the surface drifts off the readable layers, with the share of the page that's fine and the biggest good area in cm²
- the **Reader V3 ink maps**, forward and reverse (raw .tif files plus a report picture)
- a short **"check by eye" list** across all your surfaces

**It's a filter, not a text detector.** PASS means "the surface is placed right", not "there are letters here".

**What you need.** A free Kaggle account. Settings: Accelerator GPU (T4 or P100), Internet on, and under Session options Persistence "Files only" (so files survive a restart).

**How long.** Setup about 5 to 10 minutes the first time. Then about 2 to 4 minutes per surface (download plus both reading directions), a few minutes more for meshes that need rendering.

**How to use.** Run Cells 1, 2 and 3 every session. Then edit only the "Your surface" cell. Don't use "Run all".

## Beginner's guide: which surfaces pay to try

The goal of a first session: in about an hour, know which surface is worth reading, have its Reader V3 map, and know your next step. The kit handles what usually takes a week to learn by trial and error: setup, which layers hold the papyrus, which way to read, the readable window, and different scan types.

1. **Check it works (GPU on):** run Cells 1 to 3, then the "Your surface" cell as it is. It reads PHerc0841 w00, which has known letters, so you should see PASS and letters in the forward map.
2. **Check many surfaces for free (no GPU needed):** in "Many surfaces", run with `check_only=True`. You get the verdict and the map for each surface in seconds to a few minutes, with no reading.
3. **Sort:** `table()` puts PASS first, then the biggest good area. In our tests the biggest good area was the best guide to readable maps (preliminary).
4. **Spend GPU only on the best few:** turn the GPU on and `run()` the top 3 to 5. Kaggle gives about 30 free GPU hours a week.
5. **Look:** `report(name)` and `compare(name)`, and `to_check()` lists the ones that need a look by eye, with the reason.
6. **Pick your next step** from the four-box table below: fix the surface, try another surface or model, or check a blank map with Diego's IRP.

Without a GPU, steps 2 and 3 work fully. Reading (steps 1 and 4) needs the GPU.

## Cell 1. Setup (every session)
Pinned to the villa version and Reader V3 file tested on Nov 9, 2026, so later villa changes can't break it.

In [ ]:
import os, hashlib
os.chdir("/kaggle/working")
VILLA_COMMIT = "47a9c37f6f8e99afdab47bb46ad425aa726f5dc0"   # tested Nov 9, 2026
READER_V3_SHA256 = "ea405417450abd8ace1b8ff79ebce3adb646c5e48ab95da46844d77efe984d8a"
if not os.path.exists("villa"):
    !git clone -q --depth 1 --branch merge-ink-pipelines https://github.com/ScrollPrize/villa.git || echo "NO INTERNET: turn on Internet in Session options (Kaggle may ask you to verify your phone once), then restart"
!git -C villa fetch -q --depth 1 origin {VILLA_COMMIT} && git -C villa checkout -q {VILLA_COMMIT} && echo "villa pinned to {VILLA_COMMIT[:7]}"
%pip install -q -e villa/vesuvius
%pip install -q -e villa/ink-detection
%pip install -q imagecodecs awscli huggingface_hub fsspec aiohttp pandas scipy
if not os.path.exists("/kaggle/working/models/reader-v3.pth"):
    !hf download domenicor046/reader-v3 reader-v3.pth --local-dir /kaggle/working/models
h = hashlib.sha256(open("/kaggle/working/models/reader-v3.pth", "rb").read()).hexdigest()
print("Reader V3 file:", "OK, same as tested" if h == READER_V3_SHA256 else "DIFFERENT from the tested file (" + h[:12] + "...), results may differ")
!python -c "import koine_machines, vesuvius, zarr, imagecodecs, fsspec; print('session ready')"

## Cell 2. Write the kit files (every session, takes a second)
Five small files in `/kaggle/working/kit`, plus the pass line for 8.64 µm scans.

In [ ]:
import os, json
KIT = '/kaggle/working/kit'
os.makedirs(KIT, exist_ok=True)
FILES = {}
FILES['sheet_check.py'] = r'''import numpy as np
def sheet_check(v, tile=64, min_valid=0.8):
    L, H, W = v.shape; c = L // 2
    contrasts, offsets = [], []
    for y in range(0, H - tile + 1, tile):
        for x in range(0, W - tile + 1, tile):
            t = v[:, y:y+tile, x:x+tile].astype(np.float32)
            if (t > 0).all(axis=0).mean() < min_valid: continue
            p = t.reshape(L, -1).mean(axis=1)
            contrasts.append((p.max() - p.min()) / p.mean()); offsets.append(int(p.argmax()) - c)
    contrasts, offsets = np.array(contrasts), np.array(offsets)
    return {"tiles": len(offsets), "contrast_median": round(float(np.median(contrasts)), 3),
            "peak_within3": round(float((np.abs(offsets) <= 3).mean()), 3),
            "peak_edge": round(float((np.abs(offsets) >= c - 1).mean()), 3),
            "offset_median": float(np.median(offsets)), "offsets": offsets}
'''
FILES['direction.py'] = r'''import tifffile

def fr_ratio(forward_path, reverse_path, thresholds=(128, 153, 179)):
    """How much more 'ink' the forward map finds than the reverse map (capped at 100)."""
    f = tifffile.imread(forward_path)
    r = tifffile.imread(reverse_path)
    valid = (f > 0) | (r > 0)
    out = {}
    for t in thresholds:
        fa = ((f >= t) & valid).mean()
        ra = ((r >= t) & valid).mean()
        out[str(t)] = round(min(float(fa / max(ra, 1e-9)), 100.0), 2)
    return out
'''
FILES['offset_map.py'] = r'''import warnings
import numpy as np
from scipy.ndimage import generic_filter, label

def red_map(v, window, tile=64, search=7, min_valid=0.8):
    """Papyrus offset per tile (searched near the middle), smoothed, then layers outside the window."""
    L, H, W = v.shape
    c = L // 2
    lo_s, hi_s = max(c - search, 0), min(c + search + 1, L)
    m = np.full((H // tile, W // tile), np.nan)
    for i in range(H // tile):
        for j in range(W // tile):
            t = v[:, i*tile:(i+1)*tile, j*tile:(j+1)*tile].astype(np.float32)
            if (t > 0).all(axis=0).mean() < min_valid:
                continue
            p = t.reshape(L, -1).mean(axis=1)
            m[i, j] = lo_s + int(p[lo_s:hi_s].argmax()) - c
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sm = generic_filter(m, np.nanmedian, size=5, mode="constant", cval=np.nan)
    sm[np.isnan(m)] = np.nan
    lo_w, hi_w = window
    dist = np.maximum(np.maximum(lo_w - sm, sm - hi_w), 0)
    return m, dist

def map_summary(m, dist, tile=64, search=7, voxel_um=9.36):
    ok = np.isfinite(dist)
    good = ok & (dist == 0)
    lab, n = label(good)
    tile_cm2 = (tile * voxel_um * 1e-4) ** 2
    biggest = int(np.bincount(lab.ravel())[1:].max()) if n else 0
    return {"inside_window": round(float(good.sum() / max(ok.sum(), 1)), 3),
            "measured_cm2": round(float(ok.sum() * tile_cm2), 2),
            "biggest_good_cm2": round(biggest * tile_cm2, 2),
            "good_pieces": int(n),
            "search_edge": round(float((np.abs(m[ok]) >= search).mean()), 3)}

def draw_red(ax, ink, dist, tile=64, step=4):
    """Ink map in gray with strong red where the surface is outside the window (shown at 1/step size)."""
    ink = ink[::step, ::step]
    t = tile // step
    dpx = np.full(ink.shape, np.nan, dtype=np.float32)
    h, w = min(ink.shape[0], dist.shape[0] * t), min(ink.shape[1], dist.shape[1] * t)
    dpx[:h, :w] = np.kron(dist, np.ones((t, t)))[:h, :w]
    d = np.nan_to_num(dpx)
    ax.imshow(ink * 0.75, cmap="gray", vmin=0, vmax=1)
    ov = np.zeros(d.shape + (4,), dtype=np.float32)
    ov[..., 0] = 1.0; ov[..., 1] = 0.05; ov[..., 2] = 0.05
    ov[..., 3] = np.where(d >= 1, 0.55 + 0.35 * np.clip((d - 1) / 5, 0, 1), 0)
    ax.imshow(ov); ax.axis("off")
'''
FILES['render_mesh.py'] = r'''"""Render a tifxyz mesh into a surface volume (layers, height, width) for Reader V3 and the papyrus check.
Layer order and offsets match the team's published surface volumes: checked on PHerc0841 w00 and ag144
(9.366 um, 28 layers) and PHerc0800 (8.64 um, 31 layers): pixel correlation 0.96 to 0.975."""
import io, json, time
import numpy as np, tifffile, zarr, fsspec
from functools import lru_cache
from scipy.ndimage import map_coordinates

def read_tif(path):
    with fsspec.open(path, "rb") as f:
        return tifffile.imread(io.BytesIO(f.read()))

@lru_cache(maxsize=4)
def load_mesh(mesh_dir):
    X, Y, Z = [read_tif(f"{mesh_dir}/{a}.tif").astype(np.float64) for a in "xyz"]
    valid = (X > 0) & (Y > 0) & (Z > 0)
    P = np.stack([X, Y, Z], -1); P[~valid] = np.nan
    du, dv = np.gradient(P, axis=1), np.gradient(P, axis=0)
    N = np.cross(du, dv); N /= np.linalg.norm(N, axis=-1, keepdims=True)
    return P, N

def bilinear(G, r, c):
    h, w = G.shape[:2]
    r0 = np.clip(np.floor(r).astype(int), 0, h - 2); c0 = np.clip(np.floor(c).astype(int), 0, w - 2)
    fr = (r - r0)[..., None]; fc = (c - c0)[..., None]
    return (G[r0, c0] * (1 - fr) * (1 - fc) + G[r0 + 1, c0] * fr * (1 - fc)
            + G[r0, c0 + 1] * (1 - fr) * fc + G[r0 + 1, c0 + 1] * fr * fc)

def render_tile(mesh_dir, ct, scale, rows, cols, offsets, pad=4):
    P, N = load_mesh(mesh_dir)
    rr, cc = np.meshgrid(np.arange(*rows), np.arange(*cols), indexing="ij")
    p = bilinear(P, rr * scale, cc * scale); n = bilinear(N, rr * scale, cc * scale)
    n /= np.linalg.norm(n, axis=-1, keepdims=True)
    ok = np.isfinite(p).all(-1) & np.isfinite(n).all(-1)
    L = len(offsets)
    if not ok.any():
        return np.zeros((L, rr.shape[0], rr.shape[1]), np.uint8)
    pts = p[..., None, :] + np.asarray(offsets)[None, None, :, None] * n[..., None, :]
    lo = np.maximum(np.floor(pts[ok].reshape(-1, 3).min(0)).astype(int) - pad, 0)
    hi = np.ceil(pts[ok].reshape(-1, 3).max(0)).astype(int) + pad
    block = np.asarray(ct[lo[2]:hi[2], lo[1]:hi[1], lo[0]:hi[0]], dtype=np.float32)
    q = np.nan_to_num(pts - lo, nan=-1)
    coords = np.stack([q[..., 2], q[..., 1], q[..., 0]]).reshape(3, -1)
    out = map_coordinates(block, coords, order=1, mode="constant", cval=0).reshape(pts.shape[:3])
    out[~ok] = 0
    return np.clip(np.rint(out), 0, 255).astype(np.uint8).transpose(2, 0, 1)

def render_mesh(mesh_dir, ct_url, out_zarr, layers=None, tile=512):
    meta = json.load(fsspec.open(f"{mesh_dir}/meta.json").open())
    scale = meta["scale"][0]
    if layers is None:   # match the team: 28 layers at 9.36 um, 31 at 8.64 um
        layers = 31 if abs(meta.get("voxel_size_um", 9.362) - 8.64) < 0.1 else 28
    P, _ = load_mesh(mesh_dir)
    H, W = int(round(P.shape[0] / scale)), int(round(P.shape[1] / scale))
    offs = (layers - 1) / 2 - np.arange(layers)
    ct = zarr.open(ct_url, mode="r")
    grp = zarr.open_group(out_zarr, mode="w")
    try:   # zarr 2
        arr = grp.create_dataset("0", shape=(layers, H, W), chunks=(layers, 128, 128), dtype="u1",
                                 compressor=None, fill_value=0)
    except TypeError:   # zarr 3
        arr = grp.create_array("0", shape=(layers, H, W), chunks=(layers, 128, 128), dtype="u1",
                               compressors=None, fill_value=0)
    t = time.time()
    for i in range(0, H, tile):
        for j in range(0, W, tile):
            r1, c1 = min(i + tile, H), min(j + tile, W)
            arr[:, i:r1, j:c1] = render_tile(mesh_dir, ct, scale, (i, r1), (j, c1), offs)
    print(f"rendered {out_zarr}: {arr.shape} in {time.time() - t:.0f}s")
    return out_zarr
'''
FILES['preflight.py'] = r'''import argparse, json, os, subprocess, sys
import numpy as np, zarr
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tifffile
sys.path.insert(0, os.path.dirname(__file__))
from sheet_check import sheet_check
from direction import fr_ratio
from offset_map import red_map, map_summary, draw_red

MODEL = "/kaggle/working/models/reader-v3.pth"
# Pass lines = weakest known-good segment per scan type. 9.36 um from B5 (PHerc0841 w00, ag144, ag174).
# Other scan types are added to refs.json next to this file (8.64 um: set from the PHerc0009B controls).
REFS = {"9.36": {"contrast": 0.436, "peak": 0.248, "window": 0.328}}
REF_FILE = os.path.join(os.path.dirname(__file__), "refs.json")
if os.path.exists(REF_FILE):
    REFS.update(json.load(open(REF_FILE)))
WINDOWS = {"forward": (-6, 2), "reverse": (-2, 6)}   # window side follows reading direction

def open_vol(path):
    g = zarr.open(path, mode="r")
    return g["0"] if hasattr(g, "keys") and "0" in g else g

def stretch(img):
    img = img.astype(float)
    lo, hi = np.median(img), np.percentile(img, 99.5)
    return np.clip((img - lo) / max(hi - lo, 1e-9), 0, 1)

def window_share(o, lo, hi):
    return round(float(((o >= lo) & (o <= hi)).mean()), 3)

def readiness(s, direction, ref):
    return round(min(s["contrast_median"] / ref["contrast"],
                     s["peak_within3"] / ref["peak"],
                     s["window_" + direction] / ref["window"]), 3)

def score_avg(s, direction, ref):
    r = [s["contrast_median"] / ref["contrast"], s["peak_within3"] / ref["peak"],
         s["window_" + direction] / ref["window"]]
    return round(float(np.prod(r)) ** (1 / 3), 3)

FLOOR = 0.6   # any one number below 60% of its pass line still fails the surface
def verdict(avg, lowest):
    if lowest < FLOOR:
        return "FAIL"
    return "PASS" if avg >= 1.0 else ("BORDERLINE" if avg >= 0.9 else "FAIL")

ap = argparse.ArgumentParser()
ap.add_argument("--volume", required=True)
ap.add_argument("--name", required=True)
ap.add_argument("--source", choices=["published", "rendered"], default="published")
ap.add_argument("--out", default="/kaggle/working/reports")
ap.add_argument("--model", default=MODEL, help="ink model checkpoint (Reader V3 by default)")
ap.add_argument("--voxel-um", type=float, default=9.36, help="voxel size: picks the pass line and the cm2 numbers")
ap.add_argument("--force-read", action="store_true", help="read even when the verdict is FAIL (for checking the kit)")
ap.add_argument("--check-only", action="store_true", help="papyrus check, verdict and map only, no reading (works on CPU)")
a = ap.parse_args()

# 1. Data checks, and which pass line to use
arr = open_vol(a.volume)
comp = getattr(arr, "compressor", None) or getattr(arr, "compressors", None)
rep = {"name": a.name, "volume": a.volume, "shape": list(arr.shape),
       "dtype": str(arr.dtype), "compression": str(comp), "voxel_um": a.voxel_um}
rep["data_ok"] = rep["dtype"] in ("uint8", "uint16")
rep["kit_version"] = "0.1.1"
scan = min(REFS, key=lambda k: abs(float(k) - a.voxel_um))
if abs(float(scan) - a.voxel_um) > 0.3:
    rep["warning_ref"] = f"no pass line for {a.voxel_um} um scans yet: used the {scan} um one"
ref = REFS[scan]
rep["pass_line"] = {"scan_um": scan, **ref}

# 2. Direction first (it also decides the window side)
rep["direction"] = "forward" if a.source == "published" else "reverse"

# 3. Sheet check, window share both ways, scores, verdict
v = arr[:]
s = sheet_check(v)
offsets = s.pop("offsets")
for d, (lo, hi) in WINDOWS.items():
    s["window_" + d] = window_share(offsets, lo, hi)
rep["sheet"] = s
rep["readiness"] = readiness(s, rep["direction"], ref)   # score A: decides the verdict
rep["score_avg"] = score_avg(s, rep["direction"], ref)   # score B: saved for E5
rep["verdict"] = verdict(rep["score_avg"], rep["readiness"])
rep["rule"] = "B: average >= 1 PASS, >= 0.9 BORDERLINE; FAIL if any number < 0.6; direction fixed before reading"

other = "reverse" if rep["direction"] == "forward" else "forward"
if s["window_" + other] > s["window_" + rep["direction"]] + 0.05:
    rep["warning_papyrus"] = f"papyrus fits the {other} window better: check the direction"

# 3b. Offset map: where the surface sits outside the window (search +-7, smoothed 5x5)
m, dist = red_map(v, WINDOWS[rep["direction"]])
rep["map"] = map_summary(m, dist, voxel_um=a.voxel_um)   # score C numbers
MAP_FLOOR = 0.5   # less than half the page inside the window fails the surface
if rep["map"]["inside_window"] < MAP_FLOOR:
    rep["verdict"] = "FAIL"
    rep["map_fail"] = f"only {rep['map']['inside_window']:.0%} of the page is inside the readable window"

# 4. Read only if worth it (or forced), then check the direction with the ratio
os.makedirs(a.out, exist_ok=True)
stem = os.path.join(a.out, a.name)
if rep["data_ok"] and not a.check_only and (rep["verdict"] != "FAIL" or a.force_read):
    if rep["verdict"] == "FAIL":
        rep["forced_read"] = True
    subprocess.run([sys.executable, "-m", "koine_machines.inference.infer", a.volume, a.model,
                    stem + ".tif", "--overlap", "0.5", "--blend-mode", "hann",
                    "--direction", "both"], check=True)
    rep["ratio"] = fr_ratio(stem + ".tif", stem + "_reverse.tif")
    vals = list(rep["ratio"].values())
    evidence = "forward" if all(x > 1 for x in vals) else ("reverse" if all(x < 1 for x in vals) else None)
    rep["direction_rule"] = rep["direction"]
    rep["direction_evidence"] = evidence or "unclear"
    if evidence is None:
        rep["warning"] = "direction uncertain: ratio unclear, kept the rule"
        rep["next_step"] = ("map may be blank: run Diego's Ink Recoverability Protocol (IRP) to find out why, "
                            "https://github.com/dlescanogithub/vesuvius-inkcheck")
    elif evidence != rep["direction"]:
        rep["warning"] = (f"ink ratio says {evidence}, the rule says {rep['direction']}: kept {rep['direction']} "
                          "(the score never uses the other side). Check both maps by eye")
    # 5. Report image: forward, reverse, offset histogram, red map on the chosen direction
    fwd = stretch(tifffile.imread(stem + ".tif"))
    rev = stretch(tifffile.imread(stem + "_reverse.tif"))
    fig, ax = plt.subplots(1, 4, figsize=(24, 6))
    ax[0].imshow(fwd, cmap="gray"); ax[0].set_title("forward")
    ax[1].imshow(rev, cmap="gray"); ax[1].set_title("reverse")
    ax[2].hist(offsets, bins=range(-16, 17)); ax[2].set_title("papyrus offset (layers)")
    for x in ax[:2]: x.axis("off")
    draw_red(ax[3], fwd if rep["direction"] == "forward" else rev, dist)
    mp = rep["map"]
    ax[3].set_title(f"red = outside window: {mp['inside_window']:.0%} inside, "
                    f"biggest good area {mp['biggest_good_cm2']} cm2")
    forced = " (read anyway)" if rep.get("forced_read") else ""
    fig.suptitle(f"{a.name}: {rep['verdict']}{forced}, readiness {rep['readiness']}, {rep['direction']}")
    fig.savefig(stem + "_report.png", dpi=80, bbox_inches="tight")
else:
    rep["skipped"] = ("checked only, not read: run again without check_only (GPU on) to read it"
                      if a.check_only else "not read: data check or sheet check failed")

json.dump(rep, open(stem + "_report.json", "w"), indent=2)
print(json.dumps(rep, indent=2))
'''
for n, t in FILES.items():
    open(f'{KIT}/{n}', 'w').write(t)
# Pass line for 8.64 um scans: a first estimate from one segment with known letters (PHerc0009B, 125754).
# Contrast comes from that segment; peak and window keep the 9.36 um values (they measure where the papyrus sits).
json.dump({'8.64': {'contrast': 0.322, 'peak': 0.248, 'window': 0.328}}, open(f'{KIT}/refs.json', 'w'), indent=2)
print('kit files written:', sorted(os.listdir(KIT)))

## Cell 3. Helpers (every session)
Defines `run`, `report`, `compare`, `table`, `to_check` and the `add_...` commands. Nothing downloads here.

In [ ]:
# Cell 3. Helpers. Run every session, after Cells 1 and 2. Nothing downloads here.
import os, sys, json, glob, shutil, subprocess, urllib.request, warnings
import numpy as np, pandas as pd, tifffile, zarr
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Image as ShowImage, display
%matplotlib inline

W = "/kaggle/working"
KIT, VOLS, REP, KEYS = f"{W}/kit", f"{W}/vols", f"{W}/reports", f"{W}/keys"
for d in (VOLS, REP, KEYS):
    os.makedirs(d, exist_ok=True)
sys.path.insert(0, KIT)
S3 = "s3://vesuvius-challenge-open-data"
HTTP = "https://vesuvius-challenge-open-data.s3.amazonaws.com"

# Team-published surface volume per scroll
VOLUME = {
    "PHerc0841": "9.366um-1.2m-113keV-volume-20250821151531.zarr",
    "PHerc0139": "9.362um-1.2m-113keV-volume-20250728140407.zarr",
    "PHerc0009B": "8.64um-1.2m-116keV-volume-20250521125136.zarr",
    "PHerc0800": "8.64um-1.2m-116keV-volume-20250521135224.zarr",
    "PHerc0500P2": "9.362um-1.2m-113keV-volume-20250820143440.zarr",
    "PHerc0814": "9.362um-1.2m-113keV-volume-20250804134230.zarr",
}
# End of the team's answer-key preview name, per scroll (scrolls still being read have none)
KEY_TAIL = {
    "PHerc0841": "-2.403um-0.22m-77keV-volume-20260319124803-20260417190342-new_canon_autoresearch_recipe-tile256-stride128-ds8.jpg",
    "PHerc0139": "-2.399um-0.22m-78keV-volume-20260102150214-20260417190342-new_canon_autoresearch_recipe-tile256-stride128-ds8.jpg",
    "PHerc0009B": "-2.401um-0.35m-77keV-volume-20250820154339-20260417190342-new_canon_autoresearch_recipe-tile256-stride128-ds8.jpg",
    "PHerc0500P2": "-2.215um-0.4m-111keV-volume-20250526151718-20260417190342-new_canon_autoresearch_recipe-tile256-stride128-ds8.jpg",
    "PHerc0814": "-2.399um-0.22m-78keV-volume-20260309142202-20260417190342-new_canon_autoresearch_recipe-tile256-stride128-ds8.jpg",
}
UM = {"PHerc0841": 9.366, "PHerc0139": 9.362, "PHerc0009B": 8.64, "PHerc0800": 8.64,
      "PHerc0500P2": 9.362, "PHerc0814": 9.362, "PHerc0175A": 8.64, "PHerc0306B": 8.64,
      "PHerc0826": 9.362, "PHerc0813": 9.362}
# CT scans used to render meshes (add_mesh accepts one of these names or a full zarr URL)
CT = {
    "PHerc0826": "PHerc0826/volumes/20250821151701-9.362um-1.2m-113keV-masked.zarr/0",
    "PHerc0813": "PHerc0813/volumes/20250821151723-9.362um-1.2m-113keV-masked.zarr/0",
    "PHerc0175A": "PHerc0175A/volumes/20250521115057-8.640um-1.2m-116keV-masked.zarr/0",
    "PHerc0306B": "PHerc0306B/volumes/20250521133212-8.640um-1.2m-116keV-masked.zarr/0",
}

# Every surface has an entry: kind = segment | volume | mesh | patch | test
SURF = {}

def add_segment(name, scroll, segment, voxel_um=None, volume=None, group="yours"):
    """A team-published segment: scroll and segment ID as in the open data (s3://vesuvius-challenge-open-data)."""
    SURF[name] = {"kind": "segment", "scroll": scroll, "segment": segment, "group": group,
                  "volume": volume or VOLUME.get(scroll), "um": voxel_um or UM.get(scroll, 9.36), "source": "published"}
    if SURF[name]["volume"] is None:
        print(f"{name}: tell me the surface-volume folder name with volume=..., none known for {scroll}")
    return name

def add_volume(name, path, voxel_um, source="published", group="yours"):
    """Your own surface volume (zarr layer stack). path: s3://..., or a local or Kaggle dataset path.
    source="rendered" if you made it yourself with the team's vc_render_tifxyz (other layer order)."""
    if path.startswith(("http://", "https://")):
        print(f"{name}: please give the s3:// address of the same folder, or a local/Kaggle path (web addresses aren't downloaded here)")
        return None
    SURF[name] = {"kind": "volume", "path": path, "um": voxel_um, "source": source, "group": group}
    return name

def add_mesh(name, tifxyz_dir, ct, voxel_um=None, group="yours"):
    """A mesh (tifxyz folder with x.tif, y.tif, z.tif, meta.json). ct: a scroll name from CT, or a full CT zarr URL.
    The kit renders it into a layer stack first (CPU, about 1 to 2 min per few cm2)."""
    url = f"{HTTP}/{CT[ct]}" if ct in CT else ct
    um = voxel_um or UM.get(ct)
    if um is None:
        try:
            import fsspec
            um = json.load(fsspec.open(f"{tifxyz_dir}/meta.json").open()).get("voxel_size_um", 9.362)
        except Exception:
            um = 9.362
    SURF[name] = {"kind": "mesh", "mesh": tifxyz_dir, "ct": url, "um": um, "source": "published", "group": group}
    return name

# Known-letter examples (team ink labels): try the kit on these first
for n, sc, seg in [("0841_w00", "PHerc0841", "20260220213127-w00"),
                   ("0841_ag144", "PHerc0841", "20260220214732-auto_grown_20260220144552896"),
                   ("0841_ag174", "PHerc0841", "20260221022814-auto_grown_20260220174252405"),
                   ("0139_w043", "PHerc0139", "20260112000000-w043_2026011217"),
                   ("0009B_125754", "PHerc0009B", "20250919125754-auto_grown_20250919055754487_inp_hr"),
                   ("0500P2_1", "PHerc0500P2", "20250825181859--1"),
                   ("0814_46527", "PHerc0814", "20260226000000-46527_2um_try2")]:
    add_segment(n, sc, seg, group="known")
# First Letters scroll PHerc0800: 6 published segments (prize rule: don't share their maps)
for t in ["20251028213516-auto_grown_20251028213516907", "20251028220042-auto_grown_20251028220042762",
          "20251028220955-auto_grown_20251028220955262", "20251028222030-auto_grown_20251028222030940",
          "20251028225813-auto_grown_20251028225813045", "20251029010146-auto_grown_20251029010146642"]:
    add_segment("0800_" + t[8:14], "PHerc0800", t, group="eligible")
# Clean-sheet patches (Jack and Andrew Gilman, CC BY-NC), rendered here. Value = LynnHenry's verdict
PATCHES = {
    "PHerc0175A_t275_s003": "pass", "PHerc0175A_t275_s009": "pass",
    "PHerc0306B_t275_s014": "pass", "PHerc0306B_t275_s004": "fail",
    "PHerc0826B_t275_s041": "pass", "PHerc0826_t268_s007": "pass", "PHerc0826_t268_s004": "fail",
    "PHerc0813_t269_s040": "pass", "PHerc0813_t275_s055": "fail",
}
for n in PATCHES:
    sc = n.split("_")[0].replace("PHerc0826B", "PHerc0826")
    SURF[n] = {"kind": "patch", "scroll": sc, "um": UM[sc], "source": "published", "group": "patch"}

def vol(name):
    s = SURF[name]
    return s["path"] if s["kind"] == "volume" and not s["path"].startswith("s3://") else f"{VOLS}/{name}.zarr"

def _sh(cmd):
    out = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out.returncode, (out.stdout or "") + (out.stderr or "")

def get(name):
    """Make sure the surface volume is on disk: download it, use your path, or render the mesh."""
    if name not in SURF:
        print(f"{name}: unknown. Add it first with add_segment, add_volume or add_mesh"); return None
    s, p = SURF[name], vol(name)
    if os.path.exists(p):
        return p
    if s["kind"] == "segment":
        rc, msg = _sh(f"aws s3 sync --no-sign-request --only-show-errors "
                      f"{S3}/{s['scroll']}/segments/{s['segment']}/surface-volumes/{s['volume']}/ {p}")
    elif s["kind"] == "volume":
        if not s["path"].startswith("s3://"):
            print(f"{name}: path not found: {p}"); return None
        rc, msg = _sh(f"aws s3 sync --no-sign-request --only-show-errors {s['path'].rstrip('/')}/ {p}")
    elif s["kind"] in ("mesh", "patch"):
        from render_mesh import render_mesh
        if s["kind"] == "patch":
            if not os.path.exists(f"{W}/gilman"):
                _sh(f"git clone -q --depth 1 https://github.com/Ahggggg/first-letters-clean-sheets.git {W}/gilman")
            mesh, ct = f"{W}/gilman/data/{s['scroll']}/{name}.tifxyz", f"{HTTP}/{CT[s['scroll']]}"
        else:
            mesh, ct = s["mesh"], s["ct"]
        render_mesh(mesh, ct, p)
        rc, msg = 0, ""
    if rc != 0 or not os.path.exists(p):
        print(f"{name}: could not get the volume\n{msg}"); return None
    print(f"{name}: volume ready ({_sh(f'du -sh {p}')[1].split()[0]})")
    return p

def summary(name):
    f = f"{REP}/{name}_report.json"
    if not os.path.exists(f):
        print(f"{name}: no report yet"); return None
    r = json.load(open(f)); mp = r.get("map", {})
    line = (f"{name}: {r['verdict']}{' (read anyway)' if r.get('forced_read') else ''}, score {r['score_avg']}, "
            f"lowest {r['readiness']}, {r['direction']}, inside window {mp.get('inside_window')}, "
            f"biggest good area {mp.get('biggest_good_cm2')} cm2")
    if "ratio" in r: line += f", ink ratio {list(r['ratio'].values())}"
    for k in ("warning", "warning_papyrus", "warning_ref", "map_fail", "skipped", "next_step"):
        if k in r: line += f"\n    note: {r[k]}"
    print(line)
    return r

def run(name, force=False, check_only=False):
    """Run the kit on one surface. check_only=True: papyrus check, verdict and map, no reading (works on CPU).
    force=True reads it even if the verdict is FAIL (for checking only)."""
    if get(name) is None:
        return None
    s = SURF[name]
    cmd = (f"python {KIT}/preflight.py --volume {vol(name)} --name {name} "
           f"--voxel-um {s['um']} --source {s['source']}" + (" --force-read" if force else "")
           + (" --check-only" if check_only else ""))
    with open(f"{REP}/{name}_log.txt", "w") as log:
        rc = subprocess.run(cmd, shell=True, stdout=log, stderr=subprocess.STDOUT).returncode
    if rc != 0:
        print(f"{name}: ERROR. Last lines of {REP}/{name}_log.txt:")
        print("".join(open(f"{REP}/{name}_log.txt").readlines()[-15:]))
        return None
    return summary(name)

def report(name):
    png = f"{REP}/{name}_report.png"
    if os.path.exists(png):
        display(ShowImage(png))
    else:
        print(f"{name}: no report picture (not read). Use run('{name}', force=True) to read it anyway")

def answer_key(name):
    s = SURF.get(name, {})
    if s.get("kind") != "segment" or s["scroll"] not in KEY_TAIL:
        return None
    sc, seg = s["scroll"], s["segment"]
    f = f"{KEYS}/{name}.jpg"
    if not os.path.exists(f):
        try:
            urllib.request.urlretrieve(f"{HTTP}/{sc}/segments/{seg}/ink-detection/downsampled/{sc}-{seg.split('-')[0]}{KEY_TAIL[sc]}", f)
        except Exception as e:
            print(f"{name}: no answer key found ({e})"); return None
    return f

def _stretch(p):
    m = tifffile.imread(p).astype(float)
    return np.clip((m - np.median(m)) / max(np.percentile(m, 99.5) - np.median(m), 1e-9), 0, 1)

def compare(name):
    """Forward map, reverse map and the team's answer key (where one exists), side by side."""
    f, r = f"{REP}/{name}.tif", f"{REP}/{name}_reverse.tif"
    if not os.path.exists(f):
        print(f"{name}: not read yet. Use run('{name}', force=True)"); return
    k = answer_key(name)
    fig, ax = plt.subplots(1, 3, figsize=(24, 8))
    ax[0].imshow(_stretch(f), cmap="gray"); ax[0].set_title(f"{name}: forward")
    ax[1].imshow(_stretch(r), cmap="gray"); ax[1].set_title(f"{name}: reverse")
    if k:
        ax[2].imshow(Image.open(k), cmap="gray"); ax[2].set_title(f"{name}: team answer key")
    else:
        ax[2].text(0.5, 0.5, "no answer key", ha="center")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()

def table(names=None):
    """All reports as one table: PASS first, then BORDERLINE, then FAIL; biggest good area first within each."""
    rows = []
    for f in sorted(glob.glob(f"{REP}/*_report.json")):
        r = json.load(open(f)); n = r["name"]
        if names and n not in names: continue
        rows.append({"name": n, "group": SURF.get(n, {}).get("group", "?"), "scan_um": r.get("voxel_um"),
                     "verdict": r["verdict"], "read_anyway": r.get("forced_read", False),
                     "score": r["score_avg"], "lowest": r["readiness"],
                     "inside_window": r.get("map", {}).get("inside_window"),
                     "biggest_good_cm2": r.get("map", {}).get("biggest_good_cm2"),
                     "direction": r["direction"], "ink_ratio_128": r.get("ratio", {}).get("128"),
                     "lynn": PATCHES.get(n, ""),
                     "note": r.get("map_fail", r.get("warning", r.get("warning_papyrus", r.get("skipped", ""))))})
    df = pd.DataFrame(rows)
    if len(df):
        order = {"PASS": 0, "BORDERLINE": 1, "FAIL": 2}
        df = (df.assign(_o=df["verdict"].map(order)).sort_values(["_o", "biggest_good_cm2"], ascending=[True, False])
                .drop(columns="_o").reset_index(drop=True))
    return df

def to_check():
    """Short list of surfaces worth a look by eye, with the reason and what to try."""
    rows = []
    for f in sorted(glob.glob(f"{REP}/*_report.json")):
        r = json.load(open(f)); n = r["name"]; why = []
        if "ink ratio says" in r.get("warning", ""):
            why.append("ink ratio prefers the other side: compare both maps")
        if "warning_papyrus" in r:
            why.append("papyrus fits the other side: direction may be wrong")
        if r.get("direction_evidence") == "unclear":
            why.append("ink ratio unclear: map may be blank, try Diego's IRP")
        if r["verdict"] == "BORDERLINE":
            why.append("borderline: look at the map before trusting it")
        if r.get("map", {}).get("inside_window", 1) < 0.8:
            why.append("large off-papyrus areas: see panel 4")
        if r.get("forced_read"):
            why.append("FAIL but read anyway: compare the map with the verdict")
        if why:
            rows.append({"name": n, "verdict": r["verdict"], "check": "; ".join(why)})
    df = pd.DataFrame(rows)
    print(f"{len(df)} surfaces to check by eye" if len(df) else "nothing to check")
    return df

def make_shifted(src, shift=9):
    """Test tool: the same surface with the papyrus moved `shift` layers. Shows how the score reacts."""
    name = f"{src}_shift{shift}"
    if get(src) is None:
        return None
    if not os.path.exists(f"{VOLS}/{name}.zarr"):
        g = zarr.open(vol(src), mode="r")
        v = (g["0"] if hasattr(g, "keys") and "0" in g else g)[:]
        out = np.concatenate([v[shift:], np.repeat(v[-1:], shift, axis=0)])
        grp = zarr.open_group(f"{VOLS}/{name}.zarr", mode="w")
        try:
            grp.create_dataset("0", data=out, chunks=(out.shape[0], 128, 128), compressor=None)
        except TypeError:
            a = grp.create_array("0", shape=out.shape, chunks=(out.shape[0], 128, 128), dtype=out.dtype, compressors=None)
            a[:] = out
    SURF[name] = {"kind": "volume", "path": f"{VOLS}/{name}.zarr", "um": SURF[src]["um"], "source": "published", "group": "test"}
    print(f"{name}: test volume ready")
    return name

def disk():
    print(_sh(f"df -h {W} | tail -1; du -sh {VOLS}/* 2>/dev/null | sort -h")[1])

def free(name):
    """Delete one downloaded volume to make space. Its report stays. It can be downloaded again any time."""
    p = f"{VOLS}/{name}.zarr"
    if os.path.exists(p):
        shutil.rmtree(p); print(f"{name}: volume removed, report kept")

print("helpers ready:", sum(s["group"] == "known" for s in SURF.values()), "known-letter examples,",
      sum(s["group"] == "eligible" for s in SURF.values()), "PHerc0800 segments,", len(PATCHES), "patches")

## Your surface (the only cell to edit)
Pick one of the three ways to add a surface, change the name and path, then run it. The example at the bottom runs a known segment with letters, so you can see the kit working first.

In [ ]:
# 1. A team-published segment: give the scroll and segment ID (as in the open data)
# add_segment("my_seg", scroll="PHerc0139", segment="20260112000000-w043_2026011217")

# 2. Your own surface volume (zarr layer stack): an s3:// address, or a local or Kaggle dataset path
#    voxel_um picks the pass line (9.36 or 8.64). source="rendered" if you made it with vc_render_tifxyz yourself
# add_volume("my_vol", "/kaggle/input/my-dataset/my_surface.zarr", voxel_um=9.36, source="published")

# 3. A mesh (tifxyz folder) and the scroll's CT (a name like "PHerc0826", or a full CT zarr URL): the kit renders it first
# add_mesh("my_mesh", "/kaggle/input/my-dataset/my_mesh.tifxyz", ct="PHerc0826")

NAME = "0841_w00"      # change to your name, e.g. "my_seg"
run(NAME)
report(NAME)

## Try it: compare with the team's answer key
For the known-letter examples, this shows the forward map, the reverse map and the team's 2.4 µm answer key side by side.

In [ ]:
compare("0841_w00")

## Many surfaces
First check a list for free (`check_only=True`, no GPU needed), then read only the best few. You get one table (PASS first, biggest good area first) and the short list of surfaces to check by eye.
**Prize rule:** PHerc0800 and the clean-sheet patches are on First Letters scrolls. Don't share their maps before the team announces.

In [ ]:
MANY = ["0841_w00", "0841_ag144", "0841_ag174", "0139_w043"]   # or [n for n in SURF if SURF[n]["group"] == "eligible"], or list(PATCHES)
# Step 1: check all of them, no reading (works without a GPU)
for n in MANY:
    run(n, check_only=True)
display(table(MANY))

# Step 2 (GPU on): read only the best few
# for n in list(table(MANY)["name"])[:3]:
#     run(n)
# to_check()

## How to read the report

**The report picture** has four panels: the forward map, the reverse map, a histogram of where the papyrus sits in the layers, and the red map (red = the surface is outside the readable layers there, darker = further outside). The title has the verdict, the score and the direction.

**Verdicts**
- **PASS**: average score at least 1.0 (1.0 = as good as the weakest known-good example). Worth reading.
- **BORDERLINE**: average 0.9 to 1.0. Read, but check the map by eye.
- **FAIL**: average below 0.9, or any one of the three papyrus numbers below 60% of its pass line, or less than half of the page inside the readable window. Not read (use `run(name, force=True)` to read it anyway, for checking).

**The four boxes: what to do next**

| | Letters found | No letters found |
|---|---|---|
| **On the papyrus (PASS)** | A result. Look closely; on a First Letters scroll keep it private and tell the team | The surface is fine; the ink may be too faint for this reader. Try another reader, or Diego's IRP to find out why |
| **Off the papyrus (red map, FAIL)** | Treat with care: it may be reading a neighbouring sheet. Check where it sits | Fix the surface first (see the red areas), then read again |

**Limits, said plainly**
- PASS means "worth reading", not "has text".
- It can't detect a jump to the neighbouring sheet: a surface moved a whole winding still passes (in our test, a 9-layer shift read the neighbouring sheet's text).
- The papyrus scores filter but don't rank. In our pilot (23 surfaces rated by eye) they didn't predict which maps read best; the biggest good area did better (correlation about 0.8), but that is preliminary and close to Gilman's on-sheet score.
- The pass lines come from a few known segments (9.36 µm: PHerc0841 w00, ag144, ag174; 8.64 µm: one PHerc0009B segment). A held-out test is the next step.
- Works on flattened surface volumes (OME-Zarr layer stacks) and on tifxyz meshes, which it renders. Native 3D ink detection isn't supported yet.
- The direction is fixed before reading (team-published = forward). The forward/reverse ink ratio only adds a warning, because bright cracks can fool it.
- Report pictures are contrast-stretched for viewing. The raw Reader V3 output is kept as .tif next to them.

**Credits.** Reader V3 and the readable-window rule: Domenico Russo. Sheet check and direction rule: LynnHenry (sheet-contrast-check), building on rodriguescarson's sheet measure. Forward/reverse read-out: FreekCool. Clean-sheet patches: Jack and Andrew Gilman (CC BY-NC). After-reading check: Diego's Ink Recoverability Protocol (vesuvius-inkcheck). Code and data: the Vesuvius Challenge team (villa; scan data CC BY-NC 4.0). Claude helped write the code. Kit code: MIT licence.